In [1]:
from flask import Flask, request, jsonify
import cv2
import numpy as np
import base64

In [8]:
from io import BytesIO
from PIL import Image

In [2]:
app = Flask(__name__)


In [3]:
app

<Flask '__main__'>

In [7]:

def decode_image(image_data):
    image_bytes = base64.b64decode(image_data)
    image = Image.open(BytesIO(image_bytes))
    return np.array(image)

In [6]:
def encode_image(image):
    pil_img = Image.fromarray(image)
    buffer = BytesIO()
    pil_img.save(buffer, format="JPEG")
    img_str = base64.b64encode(buffer.getvalue()).decode("utf-8")
    return img_str

In [9]:
@app.route('/detect_edges', methods=['POST'])
def detect_edges():
    try:
        image_data = request.json.get('image_data')
        if not image_data:
            return jsonify({"error": "No image data provided"}), 400
        
        image = decode_image(image_data)
        
        gray_image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        
        edges = cv2.Canny(gray_image, 100, 200)
        
        result_image = encode_image(edges)
        
        return jsonify({"edges": result_image})
    
    except Exception as e:
        return jsonify({"error": str(e)}), 500

if __name__ == '__main__':
    app.run(debug=True)

 * Serving Flask app '__main__'
 * Debug mode: on


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
 * Restarting with watchdog (windowsapi)


SystemExit: 1

#  Test the API

In [14]:
import base64
import requests

def image_to_base64(file_path):
    with open(file_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')

image_base64 = image_to_base64('test_image.jpg')

response = requests.post(
    "http://127.0.0.1:5000/detect_edges",
    json={"image_data": image_base64}
)

if response.status_code == 200:
    result_image_base64 = response.json().get("edges")
    print("Edge detection successful.")
else:
    print("Error:", response.json())
